# P1-16c · mini-MIAS external set with expert masks (CPU, about 10 min)
**Runtime: CPU.** Only the mini-MIAS part of P1-16b (the DMID part is already finished and is not touched).
The images are taken first from `MyDrive/P1_v2/mias_raw/` (any mini-MIAS archive or the 322 `mdb*.pgm` files put there by hand),
otherwise from the PEIPA server. Expert masks: Verboom et al., Zenodo 10.5281/zenodo.10149914 (CC BY 4.0). Writes `MyDrive/P1_v2/mias_ext/`.

In [ ]:
# 1 · Drive, paths
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, io, json, hashlib, subprocess, zipfile
import numpy as np, pandas as pd, requests, cv2
ROOT = '/content/drive/MyDrive/QSEG_FINAL'            # read only: qseg code
EXT  = '/content/drive/MyDrive/P1_v2/dmid_ext'
for d in (f'{EXT}/images', f'{EXT}/masks'): os.makedirs(d, exist_ok=True)
assert not EXT.startswith(ROOT)
sys.path.insert(0, ROOT + '/code')
from qseg.data_prep import generate_breast_mask, resize_pair


In [ ]:
# 5 · mini-MIAS images (Drive copy first, then the PEIPA server) and the expert masks (Zenodo)
!pip -q install pynrrd
import nrrd, tarfile, shutil, requests, glob, zipfile
import matplotlib.pyplot as plt
MIAS = '/content/drive/MyDrive/P1_v2/mias_ext'
for d in (f'{MIAS}/images', f'{MIAS}/masks_expert', f'{MIAS}/masks_expert_breast', f'{MIAS}/masks_proxy'): os.makedirs(d, exist_ok=True)
MR = '/content/mias_raw'; os.makedirs(MR, exist_ok=True)
DRV = '/content/drive/MyDrive/P1_v2/mias_raw'      # put a downloaded mini-MIAS archive (zip or tar.gz) or the .pgm files here
def unpack(path, dst):
    if path.endswith('.zip'): zipfile.ZipFile(path).extractall(dst)
    elif path.endswith(('.tar.gz', '.tgz', '.tar')): tarfile.open(path).extractall(dst)
def find_pgms():
    return sorted(p for p in glob.glob(f'{MR}/**/*.pgm', recursive=True) if os.path.basename(p).lower().startswith('mdb'))
if os.path.isdir(DRV):
    for f in glob.glob(f'{DRV}/**/*', recursive=True):
        if f.lower().endswith('.pgm'): shutil.copy(f, MR)
        elif f.lower().endswith(('.zip', '.tar.gz', '.tgz', '.tar')): unpack(f, MR)
    for _ in range(2):                                      # archives inside archives (e.g. Kaggle zip containing all-mias.tar.gz)
        for f in glob.glob(f'{MR}/**/*', recursive=True):
            if f.lower().endswith(('.tar.gz', '.tgz', '.tar')) and not os.path.exists(f + '.done'):
                unpack(f, os.path.dirname(f)); open(f + '.done', 'w').close()
if len(find_pgms()) < 300:
    for url in ('http://peipa.essex.ac.uk/pix/mias/all-mias.tar.gz', 'https://peipa.essex.ac.uk/pix/mias/all-mias.tar.gz'):
        try:
            r = requests.get(url, timeout=300); r.raise_for_status()
            open(f'{MR}/all-mias.tar.gz', 'wb').write(r.content); tarfile.open(f'{MR}/all-mias.tar.gz').extractall(MR); break
        except Exception as e:
            print('failed', url, type(e).__name__)
pgms = find_pgms(); print(len(pgms), 'mini-MIAS images')
assert len(pgms) >= 300, 'mini-MIAS images not found: put the archive in MyDrive/P1_v2/mias_raw and run again'
import cv2
shapes = {cv2.imread(p, cv2.IMREAD_GRAYSCALE).shape for p in pgms[:20]}
print('image sizes', shapes); assert shapes == {(1024, 1024)}, 'expected the 1024 x 1024 mini-MIAS images'
rec = requests.get('https://zenodo.org/api/records/10149914', timeout=60).json()
for f in rec['files']:
    url = f['links'].get('self') or f['links'].get('download')
    dst = f"{MR}/{f['key']}"
    if not os.path.exists(dst): open(dst, 'wb').write(requests.get(url, timeout=600).content)
    if dst.endswith('.zip'): zipfile.ZipFile(dst).extractall(f'{MR}/masks')
nrrds = sorted(glob.glob(f'{MR}/masks/**/*.nrrd', recursive=True)); print(len(nrrds), 'expert mask files')


In [ ]:
# 6 · Align expert masks, apply the Paper 1 rule, Paper 1 preprocessing, rule-vs-expert agreement
import cv2
def dice_np(a, b):
    a, b = a > 0, b > 0; s = a.sum() + b.sum()
    return 1.0 if s == 0 else 2 * (a & b).sum() / s
import re
mask_by_id = {}
for p in nrrds:
    m = re.search(r'mdb(\d{3})', os.path.basename(p).lower())
    if m: mask_by_id['mdb' + m.group(1)] = p
rows = []
for p in pgms:
    fid = os.path.basename(p)[:-4].lower()
    if fid not in mask_by_id: rows.append(dict(file_id=fid, status='no expert mask')); continue
    gray = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    lab, _ = nrrd.read(mask_by_id[fid]); lab = np.squeeze(lab)
    k = max(3, round(20 * gray.shape[0] / 3584))            # Paper 1 kernel (20 px at 3584 px height) scaled to the image height
    proxy, thr, _ = generate_breast_mask(gray, k)
    cands = {'id': lab, 'T': lab.T, 'ud': lab[::-1], 'Tud': lab.T[::-1], 'lr': lab[:, ::-1], 'Tlr': lab.T[:, ::-1]}
    cands = {k: v for k, v in cands.items() if v.shape == gray.shape}
    best = max(cands, key=lambda k: dice_np(cands[k] > 0, proxy))
    lab = np.ascontiguousarray(cands[best])
    full = ((lab == 1) | (lab == 2)).astype(np.uint8) * 255; breast = (lab == 1).astype(np.uint8) * 255
    side = 'left' if (proxy[:, gray.shape[1] // 2:] > 0).sum() >= (proxy[:, :gray.shape[1] // 2] > 0).sum() else 'right'
    from qseg.data_prep import pad_to_square
    def prep(a, interp):
        return cv2.resize(pad_to_square(a, side), (256, 256), interpolation=interp)
    img_s = prep(gray, cv2.INTER_AREA)
    ms = {k: ((prep(v, cv2.INTER_NEAREST) > 127) * 255).astype(np.uint8) for k, v in (('proxy', proxy), ('full', full), ('breast', breast))}
    cv2.imwrite(f'{MIAS}/images/{fid}.png', img_s); cv2.imwrite(f'{MIAS}/masks_proxy/{fid}.png', ms['proxy'])
    cv2.imwrite(f'{MIAS}/masks_expert/{fid}.png', ms['full']); cv2.imwrite(f'{MIAS}/masks_expert_breast/{fid}.png', ms['breast'])
    rows.append(dict(file_id=fid, status='ok', orientation=best, has_pectoral=bool((lab == 2).any()),
                     dice_rule_vs_expert_full=round(dice_np(ms['proxy'], ms['full']), 4),
                     dice_rule_vs_expert_breast=round(dice_np(ms['proxy'], ms['breast']), 4),
                     mask_frac_expert=round(float((ms['full'] > 0).mean()), 4)))
mm = pd.DataFrame(rows); mm.to_csv(f'{MIAS}/manifest.csv', index=False)
ok = mm[mm.status == 'ok']
print('images:', len(mm), '| with expert mask:', len(ok), '| orientation used:', ok.orientation.value_counts().to_dict())
print('Paper 1 rule vs expert masks (per-image Dice), breast + pectoral:', ok.dice_rule_vs_expert_full.describe().round(3).to_dict())
print('Paper 1 rule vs expert masks (per-image Dice), breast only     :', ok.dice_rule_vs_expert_breast.describe().round(3).to_dict())


In [ ]:
# 7 · QC grid: image, expert mask (green), proxy rule (red)
fig, ax = plt.subplots(1, 6, figsize=(15, 3))
for a, fid in zip(ax, ok.file_id.sample(6, random_state=0)):
    im = cv2.imread(f'{MIAS}/images/{fid}.png', 0); rgb = cv2.cvtColor(im, cv2.COLOR_GRAY2RGB)
    for sub, col in (('masks_expert', (60, 220, 60)), ('masks_proxy', (255, 60, 60))):
        mk = cv2.imread(f'{MIAS}/{sub}/{fid}.png', 0); cs, _ = cv2.findContours(mk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(rgb, cs, -1, col, 2)
    a.imshow(rgb); a.set_title(fid, fontsize=8); a.axis('off')
plt.tight_layout(); plt.savefig(f'{MIAS}/qc.png', dpi=110); plt.show()
print('done; files in', MIAS)
